In [2]:
import pandas as pd
import numpy as np

In [3]:
movies=pd.read_csv(r'C:\Users\saurabh upadhyay\OneDrive\Desktop\movie\tmdb_5000_movies.csv')
credits=pd.read_csv(r'C:\Users\saurabh upadhyay\OneDrive\Desktop\movie\tmdb_5000_credits.csv')


In [ ]:
movies.head()

In [5]:
credits.shape

(4803, 4)

In [6]:
movies=movies.merge(credits,on='title')

In [7]:
movies.shape

(4809, 23)

In [8]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

movies.dropna(inplace=True)

In [9]:
import ast
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [10]:
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)    
def convert3(obj):
    L=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter!=3:
            L.append(i['name'])
            counter+=1
        else:
            break
    return L                                                                      

In [11]:
movies['cast']=movies['cast'].apply(convert3)
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

In [12]:
movies['crew']=movies['crew'].apply(fetch_director)
movies['overview']=movies['overview'].apply(lambda x:x.split())
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])

In [13]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [14]:
new_df=movies[['movie_id','title','tags']]

In [ ]:
new_df.head

In [16]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\saurabh upadhyay\AppData\Local\Temp\ipykernel_7156\487797088.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))


In [17]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x).lower())

C:\Users\saurabh upadhyay\AppData\Local\Temp\ipykernel_7156\3204300418.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x).lower())


In [ ]:
new_df.head


In [19]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')
new_df['tags'] = new_df['tags'].apply(lambda x: x.replace(" ", ""))
vectors = cv.fit_transform(new_df['tags']).toarray()

C:\Users\saurabh upadhyay\AppData\Local\Temp\ipykernel_7156\4079668901.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.replace(" ", ""))


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [21]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(1, np.float64(0.0)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(0.0)),
 (5, np.float64(0.0))]

In [22]:
def recommend(movie):
    movie_index=new_df[new_df['title']==movie].index[0]
    distances=similarity[movie_index]
    movies_list=sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [29]:
recommend('John Carter')

Avatar
Pirates of the Caribbean: At World's End
Spectre
The Dark Knight Rises
Spider-Man 3
